# DenseNet121 - CIFAR10 - Basic model

# Import Libraries

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
import torch.backends.cudnn as cudnn
import os


from torchvision.models import densenet121

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Import dataset

- augmented
- normalized
- padded
- shuffled
- CIFAR10

In [4]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.Resize(224),  
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.Resize(224),  
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=64, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=64, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

Files already downloaded and verified
Files already downloaded and verified


# Model : DenseNet121

In [7]:
print('create model')
model = densenet121().to(device)

num_params: int = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("number of parameters:" , num_params)

create model
number of parameters: 7978856


# Model Train and Evaluation

In [6]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr= 0.1,
                        momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=200)


In [10]:

def train(epoch):
    file_path = '../results/cifar_10/basic_model/dnesenet121_train.txt'
    
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    print('\nEpoch: %d' % epoch)
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    with open(file_path, 'a') as f:
        f.write(f'\nEpoch: {epoch}\n')
        
        for batch_idx, (inputs, targets) in enumerate(trainloader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            # progress_info = f'Loss: {train_loss / (batch_idx + 1):.3f} | Acc: {100. * correct / total:.3f}% ({correct}/{total})'
            # f.write(progress_info + '\n')

            # print(progress_info)

        epoch_summary = f'Train Summary after Epoch: {epoch}, Loss: {train_loss / len(trainloader):.3f}, Accuracy: {100. * correct / total:.3f}% ({correct}/{total})\n'
        f.write(epoch_summary)

        print(epoch_summary)


In [ ]:
def test(epoch):
    file_path = '../results/cifar_10/basic_model/densenet121_test.txt'
    
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        with open(file_path, 'a') as f:
            f.write(f'\nTesting after Epoch: {epoch}\n')
            
            for batch_idx, (inputs, targets) in enumerate(testloader):
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)

                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

                progress_info = f'Loss: {test_loss / (batch_idx + 1):.3f} | Acc: {100. * correct / total:.3f}% ({correct}/{total})'
                f.write(progress_info + '\n')

                print(progress_info)
                
            test_summary = f'Test Summary after Epoch {epoch}, Loss: {test_loss / len(testloader):.3f}, Accuracy: {100. * correct / total:.3f}% ({correct}/{total})\n'
            f.write(test_summary)
            
            print(test_summary)

In [11]:
start_epoch = 0 
for epoch in range(start_epoch, start_epoch + 200):
    train(epoch)
    test(epoch)
    scheduler.step()


Epoch: 0
Loss: 2.421 | Acc: 10.938% (14/128)
Loss: 7.004 | Acc: 10.547% (27/256)
Loss: 10.045 | Acc: 8.333% (32/384)
Loss: 15.999 | Acc: 7.422% (38/512)
Loss: 16.410 | Acc: 7.031% (45/640)
Loss: 16.696 | Acc: 7.552% (58/768)
Loss: 19.001 | Acc: 8.147% (73/896)
Loss: 20.509 | Acc: 8.301% (85/1024)
Loss: 20.245 | Acc: 8.507% (98/1152)
Loss: 19.336 | Acc: 8.984% (115/1280)
Loss: 17.992 | Acc: 8.949% (126/1408)
Loss: 16.873 | Acc: 9.505% (146/1536)
Loss: 15.860 | Acc: 9.495% (158/1664)
Loss: 15.036 | Acc: 9.431% (169/1792)
Loss: 14.336 | Acc: 9.531% (183/1920)
Loss: 13.900 | Acc: 9.668% (198/2048)
Loss: 13.505 | Acc: 9.559% (208/2176)
Loss: 12.945 | Acc: 9.722% (224/2304)
Loss: 12.415 | Acc: 9.745% (237/2432)
Loss: 11.968 | Acc: 9.609% (246/2560)
Loss: 11.552 | Acc: 9.561% (257/2688)
Loss: 11.141 | Acc: 9.446% (266/2816)
Loss: 10.827 | Acc: 9.375% (276/2944)
Loss: 10.579 | Acc: 9.408% (289/3072)
Loss: 10.254 | Acc: 9.344% (299/3200)
Loss: 9.952 | Acc: 9.315% (310/3328)
Loss: 9.669 | Acc: 